In [ ]:
import os
import subprocess
import sys
import traceback


print("=" * 60)
print("NEMOTRON LORA v40 — VALID SUBMISSION (NO TRL)")
print("Using transformers.Trainer directly (SFTTrainer removed)")
print("=" * 60)

try:
    # 1. Blackwell Environment Setup
    print("\n[1/7] Setting up Blackwell environment...")
    UTILITY_PATH = "/kaggle/usr/lib/notebooks/ryanholbrook/nvidia_utility_script"
    if os.path.exists(UTILITY_PATH):
        subprocess.run(f"tar -cf - -C {UTILITY_PATH} . | tar -xf - -C /tmp", shell=True, check=True)
        for binary in ["ptxas", "ptxas-blackwell"]:
            bin_path = f"/tmp/triton/backends/nvidia/bin/{binary}"
            if os.path.exists(bin_path):
                subprocess.run(f"chmod +x {bin_path}", shell=True, check=True)
        os.environ["TRITON_PTXAS_PATH"] = "/tmp/triton/backends/nvidia/bin/ptxas-blackwell"
        sys.path.insert(0, "/tmp")
        print("Blackwell environment initialized")
    else:
        print("WARNING: Utility script not found")

    # 2. Imports (all pre-installed, NO trl)
    print("\n[2/7] Loading dependencies...")
    import kagglehub
    import pandas as pd
    import torch
    from datasets import Dataset
    from peft import LoraConfig, TaskType, get_peft_model
    from transformers import AutoModelForCausalLM, AutoTokenizer, Trainer, TrainingArguments

    # 3. Hardware check
    print("\n[3/7] Hardware configuration:")
    if torch.cuda.is_available():
        for i in range(torch.cuda.device_count()):
            prop = torch.cuda.get_device_properties(i)
            print(f"  GPU {i}: {prop.name} ({prop.total_memory / 1024**3:.1f} GB)")

    # 4. Load competition data (ROBUST)
    print("\n[4/7] Loading training data...")
    competition_id = "nvidia-nemotron-model-reasoning-challenge"
    train_file = None
    for base_path in [f"/kaggle/input/{competition_id}", "/kaggle/input"]:
        if os.path.exists(base_path):
            for root, dirs, files in os.walk(base_path):
                for f in files:
                    if f.lower() == "train.csv":
                        train_file = os.path.join(root, f)
                        break
                if train_file:
                    break
        if train_file:
            break

    if not train_file:
        print("ERROR: Training data not found")
        sys.exit(1)

    df = pd.read_csv(train_file)
    print(f"  Columns: {list(df.columns)}")
    print(f"  Rows: {len(df)}")

    PROMPT_COL = (
        "prompt"
        if "prompt" in df.columns
        else ("question" if "question" in df.columns else "problem")
    )
    ANSWER_COL = "answer"

    # 5. Prepare training texts (simple format matching competition eval)
    print("\n[5/7] Preparing training data...")
    training_texts = []
    for idx, row in df.iterrows():
        prompt = row[PROMPT_COL]
        answer = str(row[ANSWER_COL]).strip()
        # Format: problem + reasoning + boxed answer (matches eval metric)
        text = f"Problem: {prompt}\n\nLet's solve this step by step.\n\nTherefore, the answer is \\boxed{{{answer}}}."
        training_texts.append(text)
        if (idx + 1) % 2000 == 0:
            print(f"  {idx + 1}/{len(df)}")

    print(f"  Total training texts: {len(training_texts)}")

    # 6. Load model via kagglehub (same as v20)
    print("\n[6/7] Loading Nemotron base model...")
    model_path = kagglehub.model_download(
        "metric/nemotron-3-nano-30b-a3b-bf16/transformers/default"
    )
    tokenizer = AutoTokenizer.from_pretrained(model_path, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    model = AutoModelForCausalLM.from_pretrained(
        model_path, device_map="auto", trust_remote_code=True, torch_dtype=torch.bfloat16
    )

    lora_config = LoraConfig(
        r=32,
        lora_alpha=16,
        target_modules=["in_proj", "out_proj", "up_proj", "down_proj"],
        lora_dropout=0.05,
        bias="none",
        task_type=TaskType.CAUSAL_LM,
    )
    model = get_peft_model(model, lora_config)
    model.print_trainable_parameters()

    # Tokenize
    print("  Tokenizing...")

    def tokenize_function(examples):
        return tokenizer(examples["text"], truncation=True, max_length=1024, padding="max_length")

    dataset = Dataset.from_dict({"text": training_texts})
    tokenized_dataset = dataset.map(tokenize_function, batched=True, remove_columns=["text"])
    tokenized_dataset = tokenized_dataset.map(lambda x: {"labels": x["input_ids"]}, batched=True)
    print(f"  Dataset ready: {len(tokenized_dataset)} examples")

    # 7. Training with transformers.Trainer (no trl)
    print("\n[7/7] Training LoRA adapter with transformers.Trainer...")
    training_args = TrainingArguments(
        output_dir="./nemotron_lora_adapter",
        num_train_epochs=1,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=8,
        learning_rate=1e-4,
        bf16=True,
        gradient_checkpointing=True,
        logging_steps=50,
        save_strategy="no",
        report_to="none",
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_dataset,
    )

    trainer.train()

    # Save adapter
    model.save_pretrained("./nemotron_lora_adapter")
    tokenizer.save_pretrained("./nemotron_lora_adapter")
    subprocess.run(
        "cd nemotron_lora_adapter && zip -r ../submission.zip ./*", shell=True, check=True
    )
    print("\n" + "=" * 60 + "\nSUBMISSION READY: submission.zip\n" + "=" * 60)

except Exception as e:
    print(f"\nERROR: {e}")
    traceback.print_exc()
    sys.exit(1)